### **Interrupciones Externas (ISR)(Rutina de Servicio de Interrupcion)**

## **¿Que es una Interrupcion?**

En terminos Informaticos una Interrupcion, es una suspension temporal de la ejecucion de un proceso, para pasar a ejecutar una sub-rutina, la cual por lo general, no forma parte del programa, sino que pertnece al sistema operativo o al BIOS 

Una vez finalizada esta sub-rutina, la ejecucion del programa continua por donde se habia quedado 

Para entender la necesidad y utilidad de las interrupciones debemos enfocarnos en la manera de trabajar que tienen la mayoria de microcontroladores basados en arduino, ya que estos tienen un flujo de trabajo ciclico y dentro de este ciclo bucle o tambien conocido como void loop(), es donde se lleva a cabo la logica de la programacion del microcontrolador 



### Recrearemos el funcionamiento de un semaforo, que tenga un pulsador de emergencia que nos permita encender todas las lamparas del semaforo a la vez

```bash
void loop(){
    digitalWrite(luz_vewrde,HIGH);
    digitalWrite(luz_roja,LOW);
    delay(2000);
    digitalWrite(luz_amarilla,HIGH);
    digitalWrite(luz_verde,LOW);
    delay(2000);
    digitalWrite(luz_roja,HIGH);
    digitalWrite(luz_amarilla,HIGH);
    delay(2000);

    int estadoPulsador = digitalWrite(pulsador); //INTERRUPCION

    if(estadoPulsador == LOW){
        encenderTodas();
    }
}
```

Este tipo de programacion **presenta el inconveniente de que la interrupcion no se realizara hasta que el algoritmo llegue a esa linea**, y si se requiere una interrupcion de inmediato al ejecutar el algoritmo, no sera posible, dado que la interrupcion se encuentra mucho mas abajo


## **Existen dos tipos principales de interrupciones**

- #### **Interrupciones Externas**: 

Tambien conocidas como Interrupciones de hardware, son las queocurren en respuesta a un evento externo, como por ej: el cambio de estado de un GPIO al presionar un boton o la deteccion de un evento tactil en algguna de sus entradas  

- #### **Interrupciones Internas**:

Tambien conocidas como interrupciones de software, son las que ocurren en respuesta a un evento programado, como por ej: cuando finaliza el contador de un temporizador o cuando ocurre un error por desbordamiento o por excepciones en el programa

### **Ahora mismo nos centraremos en las interrupciones externas**

#### Todos los pines de la placa ESP32, pueden usarse como pines de interrupcion

Como ya dijimos, una interrupcion es un evento que interrumpe el flujo principal del programa para pasar a ejecutar una porcion especial de cpdigo (sub-rutina)

Pues justo lo que necesitamos para crear una interrupcion es una **funcion** que se llamara cuando un pin seleccionado cambie el valor de su señal

Esta funcion es conocida como **Rutina de Servicio de Interrupcion** y sera la porcion especial de codigo que se ejecutara cuando la interrupcion tenga lugar

### - **El IDE de arduino nos propórciona una funcion que se llama**

``` bash 
attachInterrupt(GPIO,ISR,MODO) 
```
que nos servira para asociar una interrupcion, con un pin seleccionado

**Esta funcion recibe tres parametros:**

- GPIO: (pin de interrupcion), el GPIO asociado con el pin que provocara que ocurra una interrupcion (Le dice al ESP32 que pin debe monitorear)

- ISR: (funcion que se llamara cada vez que se dispare la interrupcion), Nommbre de la funcion asociaada a la interrupcion

- MODO: Indica cuando debe ejecutarse la interrupcion (existen 5 Modos diferentes)

  - LOW: este modo ejecutara la interrupcion, cuando el pin se encuentre en estado bajo
  - HIGH: este modo ejecutara la interrupcion cuando el pin se encuentre en estado alto 
  - CHANGE: el modo change ejecutara la interrupcion cunando se produzca un cambio de estado, tanto de HIGH a LOW como de LOW a HIGH
  - FALLING: este modo ejecutara la interrupcion cuando el pin pase de estado Alto a estado Bajo
  - RISING: este modo ejecutara la interrupcion cuando el pin pase de estado Bajo a estado Alto

### - **Tambien disponemos de otra funcion que nos permite desactivar una interrupcion existente**

``` bash 
detachInterrupt(GPIO) 
```
Esta funcion recibe un unico parametro, que corresponde con el pin GPIO asociado a la interrupcion 


## **Teniendo todo esto en cuenta volvemos al ejemplo del semaforo**

Veamos como solucionar el problema que teniamos a la hora de ejecutar una interrupcion, 

La solucion es incorporar llamados a la funcion attachInterrupt dentro del **void setup(){}**

```bash
int luz_verde = 14;
int luz_amarilla = 27;
int luz_roja = 26;
int pulsador = 25;

void IRAM_ATTR encenderTodas(){
    digitalWrite(luz_verde, HIGH); 
    digitalWrite(luz_amarilla, HIGH);
    digitalWrite(luz_roja, HIGH);
}

void setup(){
    pinMode(luz_verde, OUTPUT);
    pinMode(luz_amarilla, OUTPUT);
    pinMode(luz_roja, OUTPUT);
    pinMode(pulsador, INPUT_PULLUP);
    attachInterrupt(digitaLPinToInterrupt(pulsador), encenderTodas, RISING); //attachInterrupt(GPIO,ISR,MODO) 
}

void loop(){
    digitalWrite(luz_vewrde,HIGH);
    digitalWrite(luz_amarilla,LOW);
    digitalWrite(luz_roja,LOW)
    delay(2000);
    digitalWrite(luz_amarilla,HIGH);
    digitalWrite(luz_verde,LOW);
    digitalWrite(luz_roja,LOW)
    delay(2000);
    digitalWrite(luz_roja,HIGH);
    digitalWrite(luz_amarilla,HIGH);
    digitalWrite(luz_verde,LOW)
    delay(2000);
}
```

### **OBS:** 

El atributo **IRAM_ATTR** delante del nombre de una funcion, hace que el bloque del codigo de la funcion sea colocada en la memoria RAM del ESP32, 

De lo contrario el codigo se coloca en la memoria flash, y la memoria flash es mucho mas lenta que la memoria RAM interna 

Y como generalmente cuando hacemos uso de interrupciones, queremos que se ejecute lo mas rapido posible

## **Detalles a tener en cuenta**

Como hemos visto, las interrupciones externas son sencillas de usar, pero debemos tener en cuenta una serie de **normas y buenas practicas** a la hora de utilizar interrupciones en el ESP32

- La interrupcion (ISR) debe tener un tiempo de ejecucion lo mas corto posible, 

(ya que mientras se este ejecutando, todas las demasa partes del bucle principal, permanecen detenidas)

- Las funciones de interrupcion deben tener el atributo **IRAM_ATTR**, de acuerdo con la documentacion del ESP32 

(con el objetivo principal de que su ejecucuion sea lo mas agil posible)

- Las ISR no reciben parametros ni devuelven ningun valor 

- La unica forma de compartir datos entre la funcion de interrupcion y el programa principal es a traves de variables declaradas como volatiles
 
 **Ej:** 
 ```bash
 volatile int state = LOW;
 ```
 
 Las variables declaradas como volatiles, cargan su valor desde la memoria RAM y no desde un registro de almacenamiento

 Lo que nos garantiza que el valor devuelto sea un valor exacto

### **CONCLUSION**

Las interrupciones son un mecanismo muy potente y comodo que mejora nuestros programas y nos permite realizar acciones que no serian posibles sin el uso de interrupciones

Sin duda son una gran opcion que debemos añadir a nuestra caja de herramientas 